# 统一清洗：发票 CSV → 回归数据

**输入文件**（均位于 `G:\Kuangyu_Temp\Outsource\productivity\`）：

| 文件 | 内容 | 颗粒度 | 口径 |
|---|---|---|---|
| `firm_buy.csv` | 样本企业采购 | 购方企业ID × 购方地区 × 项目代码 | 样本 |
| `firm_sell.csv` | 样本企业销售 | 销方企业ID × 销方地区 × 项目代码 | 样本 |
| `city_buy.csv` | 全城市采购 | 购方地区 × 项目代码 | **全量** |
| `city_sell.csv` | 全城市销售（供给侧，按购方地区汇总） | 购方地区 × 项目代码 | **全量** |
| `bianma.dta` | 合法 9 位产品码（2778 个） | — | — |
| `full_data.dta` | 企业特征 | firm × year | — |

**输出文件**（落在同目录）：

| 文件 | 内容 |
|---|---|
| `invoice_panel.dta` | 主回归面板：firm × product × city × year，DV = `ln_p_net` |
| `market_conds.dta` | 市场条件：product × city × year（买方/卖方均来自全量） |
| `firm_chars.dta` | 企业特征：firm × year |

**关键设计**：
- `city` 统一 = `购方地区`（买方所在城市，全程一致）
- `firm_buy.csv` 直接带 `购方地区`，不再需要额外的 firm_city 对照表
- `city_sell.csv` 按 `购方地区` 汇总卖方企业数，和买方口径对称
- 市场条件（`n_buyers`, `n_sellers`, `p_mkt` 等）全部来自全量数据

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

BASE = Path(r'G:\Kuangyu_Temp\Outsource\productivity')
ROOT = BASE.parent

print('BASE:', BASE)
print('ROOT:', ROOT)

## Step 1　读入所有 CSV

ID、地区代码、项目代码全部按字符串读入，避免长数字被截断。

In [ ]:
NA = ['', 'NULL', '(Null)', 'null', 'None', 'nan', 'NaN', '#N/A']

firm_buy  = pd.read_csv(BASE / 'firm_buy.csv',  dtype=str, na_values=NA, encoding='utf-8-sig')
firm_sell = pd.read_csv(BASE / 'firm_sell.csv', dtype=str, na_values=NA, encoding='utf-8-sig')
city_buy  = pd.read_csv(BASE / 'city_buy.csv',  dtype=str, na_values=NA, encoding='utf-8-sig')
city_sell = pd.read_csv(BASE / 'city_sell.csv', dtype=str, na_values=NA, encoding='utf-8-sig')

for name, df in [('firm_buy', firm_buy), ('firm_sell', firm_sell),
                 ('city_buy', city_buy), ('city_sell', city_sell)]:
    df.columns = df.columns.str.strip()
    print(f'{name:12s}  shape={df.shape}  cols={list(df.columns)}')

## Step 2　统一列名

`firm_buy` 直接带 `购方地区`，后续不再需要 firm_city 对照表。

In [ ]:
fb = firm_buy.rename(columns={
    '购方企业ID': 'firm_id', '购方地区': 'city',
    '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'
})[['firm_id', 'city', 'product_code', 'value', 'qty']].copy()

fs = firm_sell.rename(columns={
    '销方企业ID': 'firm_id', '销方地区': 'city',
    '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'
})[['firm_id', 'city', 'product_code', 'value', 'qty']].copy()

# city_buy: 全量采购侧，含买方企业数
cb_cols = {'购方地区': 'city', '项目代码': 'product_code',
           '买方企业数': 'n_buyers_raw', '金额合计': 'value', '数量合计': 'qty'}
cb = city_buy.rename(columns=cb_cols)[list(cb_cols.values())].copy()

# city_sell: 全量供给侧（按购方地区汇总），含卖方企业数
cs_cols = {'购方地区': 'city', '项目代码': 'product_code',
           '卖方企业数': 'n_sellers_raw', '金额合计': 'value', '数量合计': 'qty'}
cs = city_sell.rename(columns=cs_cols)[list(cs_cols.values())].copy()

for name, df in [('fb', fb), ('fs', fs), ('cb', cb), ('cs', cs)]:
    print(f'{name}  {df.shape}')

## Step 3　转换数值、清洗项目代码

对四个数据集统一执行：
1. value / qty 转数值（`errors='coerce'`）
2. 项目代码必须纯数字
3. 右补零到 19 位，取前 9 位 → `product_id`
4. 删除 value ≤ 0 或 qty ≤ 0 的行（含红冲净值为负的记录）

In [ ]:
def clean_codes(df, label):
    n0 = len(df)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['qty']   = pd.to_numeric(df['qty'],   errors='coerce')
    # 纯数字项目代码
    df = df.dropna(subset=['product_code', 'value', 'qty']).copy()
    df = df[df['product_code'].str.fullmatch(r'\d+', na=False)].copy()
    # 右补零 → 19 位 → 取前 9 位
    df['product_id'] = df['product_code'].str.ljust(19, '0').str[:9]
    # 删除非正
    df = df[(df['value'] > 0) & (df['qty'] > 0)].copy()
    print(f'{label:6s}: {n0:>10,} → {len(df):>10,}  unique products: {df["product_id"].nunique()}')
    return df

fb = clean_codes(fb, 'fb')
fs = clean_codes(fs, 'fs')
cb = clean_codes(cb, 'cb')
cs = clean_codes(cs, 'cs')

## Step 4　匹配 bianma — 只保留合法 9 位产品码

In [ ]:
bianma = pd.read_stata(ROOT / 'bianma.dta')
valid = set(bianma['product_id'].astype(str).str.strip())
print(f'bianma: {len(valid)} valid products')

for name, df in [('fb', fb), ('fs', fs), ('cb', cb), ('cs', cs)]:
    n = len(df)
    df = df[df['product_id'].isin(valid)].copy()
    print(f'{name}: {n:>10,} → {len(df):>10,}  match {len(df)/n*100:.1f}%'
          f'  unique products: {df["product_id"].nunique()}')
    if name == 'fb': fb = df
    if name == 'fs': fs = df
    if name == 'cb': cb = df
    if name == 'cs': cs = df

## Step 5　构造 DV：企业采购价格面板

`firm_buy` 已经是企业 × 购方地区 × 产品的年度聚合。这里再聚合一次以防重复行，同时加 year。

**DV = `p_buy` = 金额合计 / 数量合计**（用`ln_p_net`作为别名，保持与Stata脚本兼容）。

In [ ]:
fb['year'] = 2017

dv = fb.groupby(['firm_id', 'product_id', 'city', 'year'], as_index=False).agg(
    value  = ('value', 'sum'),
    qty    = ('qty',   'sum'),
    n_rows = ('value', 'size')
)
dv = dv[(dv['value'] > 0) & (dv['qty'] > 0)].copy()

dv['p_buy']    = dv['value'] / dv['qty']
dv['ln_p_buy'] = np.log(dv['p_buy'])
dv['ln_p_net'] = dv['ln_p_buy']   # 兼容 Stata 脚本
dv['p_net']    = dv['p_buy']
dv['ln_qty']   = np.log(dv['qty'])

print('invoice_panel rows:', len(dv))
print('unique firms:   ', dv['firm_id'].nunique())
print('unique products:', dv['product_id'].nunique())
print('unique cities:  ', dv['city'].nunique())
dv[['p_buy', 'qty', 'value']].describe(percentiles=[.01, .05, .5, .95, .99]).round(2)

## Step 6　市场条件（买方侧）

来自全量 `city_buy.csv`，按 `city × product_id × year` 聚合。

- `mkt_value`, `mkt_qty`, `p_mkt`：全市场采购金额、数量、均价
- `n_buyers`：买方企业数（来自 city_buy 的 `买方企业数`，因已按 19→9 位合并，可能轻微重复计数，参见 city_market_condition_note.md §7.1）

In [ ]:
cb['year']          = 2017
cb['n_buyers_raw']  = pd.to_numeric(cb['n_buyers_raw'], errors='coerce')

mkt_buy = cb.groupby(['product_id', 'city', 'year'], as_index=False).agg(
    mkt_value  = ('value',        'sum'),
    mkt_qty    = ('qty',          'sum'),
    n_buyers   = ('n_buyers_raw', 'sum'),
    n_raw_buy  = ('product_code', 'nunique')
)
mkt_buy = mkt_buy[(mkt_buy['mkt_value'] > 0) & (mkt_buy['mkt_qty'] > 0)].copy()
mkt_buy['p_mkt']       = mkt_buy['mkt_value'] / mkt_buy['mkt_qty']
mkt_buy['ln_p_mkt']    = np.log(mkt_buy['p_mkt'])
mkt_buy['ln_mkt_qty']  = np.log(mkt_buy['mkt_qty'])
mkt_buy['ln_n_buyers'] = np.log(mkt_buy['n_buyers'].clip(lower=1))

print('mkt_buy rows:', len(mkt_buy))
print('unique products:', mkt_buy['product_id'].nunique())
print('unique cities:  ', mkt_buy['city'].nunique())
mkt_buy[['mkt_qty', 'p_mkt', 'n_buyers']].describe(percentiles=[.01, .5, .99]).round(2)

## Step 7　市场条件（卖方侧）

来自全量 `city_sell.csv`，按**购方地区** × 产品汇总（不是销方地区）。

经济含义：`n_sellers` = 在买家所在城市 c，有多少不同供应商曾向这里的买家供货 product p。
这和 `n_buyers` 口径对称，merge 键也统一。

In [ ]:
cs['year']           = 2017
cs['n_sellers_raw']  = pd.to_numeric(cs['n_sellers_raw'], errors='coerce')

mkt_sell = cs.groupby(['product_id', 'city', 'year'], as_index=False).agg(
    sell_value  = ('value',          'sum'),
    sell_qty    = ('qty',            'sum'),
    n_sellers   = ('n_sellers_raw',  'sum'),
    n_raw_sell  = ('product_code',   'nunique')
)
mkt_sell = mkt_sell[(mkt_sell['sell_value'] > 0) & (mkt_sell['sell_qty'] > 0)].copy()
mkt_sell['ln_n_sellers'] = np.log(mkt_sell['n_sellers'].clip(lower=1))

print('mkt_sell rows:', len(mkt_sell))
print('unique products:', mkt_sell['product_id'].nunique())
print('unique cities:  ', mkt_sell['city'].nunique())
mkt_sell[['sell_qty', 'n_sellers']].describe(percentiles=[.01, .5, .99]).round(2)

## Step 8　合并市场条件

以买方侧为基础（`mkt_buy`），left join 卖方侧（`mkt_sell`）。缺失的 `n_sellers` 在后面诊断时会看到比例。

In [ ]:
market_conds = mkt_buy.merge(
    mkt_sell[['product_id', 'city', 'year', 'sell_value', 'sell_qty', 'n_sellers', 'ln_n_sellers']],
    on=['product_id', 'city', 'year'],
    how='left'
)

print('market_conds rows:', len(market_conds))
print('n_sellers missing:', market_conds['n_sellers'].isna().mean())

## Step 9　企业特征（来自 full_data.dta）

分块读取避免 OOM。只取 firm × year 级别的变量：规模、产品数、是否中介。

In [ ]:
usecols = ['firm_id', 'year', 'firm_total_output', 'firm_total_outsource',
           'n_products', 'is_intermediary']

parts = []
for ch in pd.read_stata(ROOT / 'full_data.dta', columns=usecols, chunksize=500_000):
    parts.append(ch.drop_duplicates(['firm_id', 'year']))

firm_chars = pd.concat(parts, ignore_index=True).drop_duplicates(['firm_id', 'year'])
firm_chars['firm_id'] = firm_chars['firm_id'].astype(str).str.strip()
firm_chars['year']    = firm_chars['year'].astype(int)
firm_chars['ln_firm_output']    = np.log(firm_chars['firm_total_output'].clip(lower=1))
firm_chars['ln_firm_outsource'] = np.log(firm_chars['firm_total_outsource'].clip(lower=1))

print('firm_chars rows:', len(firm_chars))
print('unique firms:   ', firm_chars['firm_id'].nunique())

## Step 10　覆盖率诊断

检查三个 merge 的覆盖率，确认没有大规模样本损失。

In [ ]:
# DV × 市场条件
tmp = dv.merge(
    market_conds[['product_id', 'city', 'year', 'ln_p_mkt', 'ln_n_buyers', 'ln_n_sellers']],
    on=['product_id', 'city', 'year'], how='left', indicator=True
)
print('DV × market_conds:')
print(tmp['_merge'].value_counts())
print(f'  market missing: {(tmp["_merge"] != "both").mean()*100:.1f}%')
print(f'  ln_n_sellers missing among matched: {tmp["ln_n_sellers"].isna().mean()*100:.1f}%')

# DV × 企业特征
tmp2 = dv.merge(
    firm_chars[['firm_id', 'year', 'ln_firm_output', 'is_intermediary']],
    on=['firm_id', 'year'], how='left', indicator=True
)
print('\nDV × firm_chars:')
print(tmp2['_merge'].value_counts())
print(f'  firm_chars missing: {(tmp2["_merge"] != "both").mean()*100:.1f}%')

## Step 11　导出 Stata 文件

In [ ]:
# invoice_panel
inv_cols = ['firm_id', 'product_id', 'city', 'year',
            'value', 'qty', 'p_buy', 'ln_p_buy', 'p_net', 'ln_p_net', 'ln_qty', 'n_rows']
dv[inv_cols].to_stata(BASE / 'invoice_panel.dta', write_index=False, version=118)
print('invoice_panel.dta:', len(dv))

# market_conds
mkt_cols = ['product_id', 'city', 'year',
            'mkt_value', 'mkt_qty', 'p_mkt', 'ln_p_mkt', 'ln_mkt_qty',
            'n_buyers', 'ln_n_buyers',
            'sell_value', 'sell_qty', 'n_sellers', 'ln_n_sellers']
market_conds[mkt_cols].to_stata(BASE / 'market_conds.dta', write_index=False, version=118)
print('market_conds.dta: ', len(market_conds))

# firm_chars
fc_cols = ['firm_id', 'year', 'firm_total_output', 'firm_total_outsource',
           'n_products', 'is_intermediary', 'ln_firm_output', 'ln_firm_outsource']
firm_chars[fc_cols].to_stata(BASE / 'firm_chars.dta', write_index=False, version=118)
print('firm_chars.dta:   ', len(firm_chars))

print('\n=== 清洗完成 ===')